In [1]:
import pandas as pd

In [2]:
raw_stats = pd.read_csv("../data/raw/team_stats.csv", index_col=0)
raw_sched = pd.read_csv("../data/raw/team_schedules.csv", index_col=0)
print(raw_stats.shape, raw_sched.shape)

(3386, 103) (1693, 46)


In [3]:
STAT_KEEP = [
    "season", "week", "team", "opponent_team",
    # Passing
    "completions", "attempts", "passing_yards", "passing_tds",
    "passing_interceptions", "sacks_suffered", "sack_yards_lost",
    "passing_air_yards", "passing_yards_after_catch",
    "passing_first_downs", "passing_epa", "passing_cpoe",
    # Rushing
    "carries", "rushing_yards", "rushing_tds", "rushing_first_downs", "rushing_epa",
    # Defense
    "def_tackles_solo", "def_tackles_for_loss", "def_sacks", "def_qb_hits",
    "def_interceptions", "def_interception_yards", "def_pass_defended",
    "def_fumbles_forced",
    # Penalties & timeouts
    "penalties", "penalty_yards", "timeouts",
    # Returns
    "punt_returns", "punt_return_yards", "kickoff_returns", "kickoff_return_yards",
    # Kicking
    "fg_made", "fg_att", "fg_long", "fg_pct",
]

stats = raw_stats[STAT_KEEP].copy()
stats.shape

(3386, 40)

In [4]:
SCHED_KEEP = [
    "season", "week", "game_type",
    "away_team", "away_score", "home_team", "home_score",
    "overtime", "away_rest", "home_rest",
    "div_game", "roof", "surface", "temp", "wind",
]

sched = raw_sched[SCHED_KEEP].dropna(subset=["away_score", "home_score"]).copy()
sched.head()

,season,week,game_type,away_team,away_score,home_team,home_score,overtime,away_rest,home_rest,div_game,roof,surface,temp,wind
0,2020,1,REG,HOU,20,KC,34,0,7,7,0,outdoors,grass,56.0,7.0
1,2020,1,REG,SEA,38,ATL,25,0,7,7,0,closed,fieldturf,NaN,NaN
2,2020,1,REG,CLE,6,BAL,38,0,7,7,1,outdoors,grass,76.0,5.0
3,2020,1,REG,NYJ,17,BUF,27,0,7,7,1,outdoors,astroturf,67.0,15.0
4,2020,1,REG,LV,34,CAR,30,0,7,7,0,outdoors,grass,81.0,5.0


In [5]:
GAME_COLS = ["season", "week", "overtime", "div_game", "roof", "surface", "temp", "wind"]

home = sched.copy()
home["team"] = home["home_team"]
home["opponent_team"] = home["away_team"]
home["is_home"] = True
home["won"] = home["home_score"] > home["away_score"]
home["rest"] = home["home_rest"]
home["opp_rest"] = home["away_rest"]

away = sched.copy()
away["team"] = away["away_team"]
away["opponent_team"] = away["home_team"]
away["is_home"] = False
away["won"] = away["away_score"] > away["home_score"]
away["rest"] = away["away_rest"]
away["opp_rest"] = away["home_rest"]

TEAM_SCHED_COLS = GAME_COLS + ["game_type", "team", "opponent_team", "is_home", "won", "rest", "opp_rest"]
sched_long = pd.concat([home[TEAM_SCHED_COLS], away[TEAM_SCHED_COLS]])
sched_long.head()

,season,week,overtime,div_game,roof,surface,temp,wind,game_type,team,opponent_team,is_home,won,rest,opp_rest
0,2020,1,0,0,outdoors,grass,56.0,7.0,REG,KC,HOU,True,True,7,7
1,2020,1,0,0,closed,fieldturf,NaN,NaN,REG,ATL,SEA,True,False,7,7
2,2020,1,0,1,outdoors,grass,76.0,5.0,REG,BAL,CLE,True,True,7,7
3,2020,1,0,1,outdoors,astroturf,67.0,15.0,REG,BUF,NYJ,True,True,7,7
4,2020,1,0,0,outdoors,grass,81.0,5.0,REG,CAR,LV,True,False,7,7


In [6]:
df = stats.merge(
    sched_long,
    on=["season", "week", "team", "opponent_team"],
    how="inner"
)

df[["team", "opponent_team", "is_home", "won", "rest", "opp_rest"]].head()

,team,opponent_team,is_home,won,rest,opp_rest
0,ARI,SF,False,True,7,7
1,ATL,SEA,True,False,7,7
2,BAL,CLE,True,True,7,7
3,BUF,NYJ,True,True,7,7
4,CAR,LV,True,False,7,7


In [7]:
STAT_COLS = [c for c in STAT_KEEP if c not in ("season", "week", "team", "opponent_team")]

opp = df[["season", "week", "team", "opponent_team"] + STAT_COLS].copy()
opp.columns = ["season", "week", "opp_team", "opp_opponent"] + [f"opp_{c}" for c in STAT_COLS]

full = df.merge(
    opp,
    left_on=["season", "week", "team", "opponent_team"],
    right_on=["season", "week", "opp_opponent", "opp_team"],
    how="inner"
).drop(columns=["opp_team", "opp_opponent"])

full.shape

(3386, 87)

In [8]:
full = full[full["game_type"] == "REG"].drop(columns="game_type")
full = full.sort_values(["season", "week", "team"]).reset_index(drop=True)
full.head()

,season,week,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,...,opp_penalty_yards,opp_timeouts,opp_punt_returns,opp_punt_return_yards,opp_kickoff_returns,opp_kickoff_return_yards,opp_fg_made,opp_fg_att,opp_fg_long,opp_fg_pct
0,2020,1,ARI,SF,26,40,230,1,1,2,...,53,4,3,29,1,16,2,2,52.0,1.0
1,2020,1,ATL,SEA,37,54,450,2,1,2,...,46,4,1,15,2,43,1,1,42.0,1.0
2,2020,1,BAL,CLE,21,26,284,3,0,2,...,80,1,1,1,0,0,0,1,NaN,0.0
3,2020,1,BUF,NYJ,33,46,312,2,0,3,...,95,5,0,0,2,32,1,1,31.0,1.0
4,2020,1,CAR,LV,22,34,269,1,0,1,...,40,4,2,37,0,0,2,2,54.0,1.0


In [9]:
full.to_csv("../data/processed/team_stats_processed.csv", index=False)
print(f"Saved {len(full)} rows, {len(full.columns)} columns")

Saved 3230 rows, 86 columns
